In [11]:
import pandas as pd
import os
import random
from dotenv import load_dotenv
from openai import OpenAI



DATA_PATH='../../data/'

CONTEXT_REL_CONFIG = {
    "name": "gpt-5.6-luna",
    "reasoning_effort": "low",
}

load_dotenv(override=True)

model_config = CONTEXT_REL_CONFIG

MODEL_NAME = model_config["name"]
REASONING_EFFORT = model_config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")



df=pd.read_csv(f'{DATA_PATH}df_n1.csv')



Using model: gpt-5.6-luna with reasoning effort: low


In [12]:
import json
import pandas as pd


# ============================================================
# Configuration
# ============================================================

INPUT_DATA_FILE = "../../data/df_n1.csv"
OUTPUT_JSONL_FILE = "infile_context_relevance_batch.jsonl"


# ============================================================
# Prompt construction
# ============================================================

def build_context_messages(row):

    system_prompt = """
You are an expert software engineer performing context compression for an automated code review system.

Your task is NOT to summarize the file and NOT to reproduce the original file.
Your task is to extract the MINIMUM amount of source code required for another LLM to correctly review the given code hunk.

You will receive:
1. The original source file (<old_file>)
2. The changed code hunk (<hunk>)

Your goal:
Produce a compact code context that preserves only the information necessary to understand the behavior, dependencies, and potential issues of <hunk>.

Selection strategy:
- Start by including only the changed hunk.
- Add the smallest enclosing scope needed (function, method, or class).
- Add external definitions only when the hunk depends on them and their absence would make the behavior ambiguous.
- Add called functions, variables, attributes, imports, decorators, or parent classes only when they directly influence the logic of the hunk.
- Prefer short relevant snippets over complete files.
- If a definition is large, include only the relevant parts.

Strict exclusion rules:
- Do NOT return the entire file if it's loo large.
- Do NOT include file headers, licenses, comments, documentation, or unrelated code.
- Do NOT include neighboring functions unless they are required to understand the hunk.
- Do NOT include imports unless they are necessary to understand a referenced component.
- Do NOT include boilerplate code.

Think like a human reviewer:
A reviewer does not read the whole repository file. They inspect the changed code and only open the definitions required to reason about correctness.

Output requirements:
- Return ONLY the extracted source code.
- No explanations.
- No markdown fences.
- No comments about your selection process.

The final output should be significantly smaller than <old_file>.
"""

    user_prompt = f"""
<old_file>
{row["oldf"]}
</old_file>


<hunk>
{row["hunk"]}
</hunk>
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


# ============================================================
# Build one Batch API request
# ============================================================

def build_batch_request(row, index):

    messages = build_context_messages(row)

    request = {
        "custom_id": f"row_{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL_NAME,
            "messages": messages
        }
    }

    if REASONING_EFFORT:
        request["body"]["reasoning_effort"] = REASONING_EFFORT

    return request


# ============================================================
# Construct JSONL
# ============================================================

def build_context_batch_jsonl(df, output_file):

    total = len(df)
    written = 0
    skipped = 0

    print(f"[START] Building JSONL for {total} rows")

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        for index, row in df.iterrows():

            # Same condition as in the original API pipeline
            if (
                pd.isna(row["hunk"])
                or pd.isna(row["oldf"])
            ):
                print(
                    f"[SKIP] Row {index}: "
                    "missing old file or hunk"
                )

                skipped += 1
                continue

            request = build_batch_request(
                row,
                index
            )

            f.write(
                json.dumps(
                    request,
                    ensure_ascii=False
                )
                + "\n"
            )

            written += 1

            if written % 10 == 0:
                print(
                    f"[PROGRESS] "
                    f"{written}/{total} requests written"
                )

    print("\n[DONE] JSONL construction")
    print(f"[INFO] Requests written: {written}")
    print(f"[INFO] Rows skipped: {skipped}")
    print(f"[INFO] Output: {output_file}")




In [13]:
build_context_batch_jsonl(
    df,
    OUTPUT_JSONL_FILE
)

[START] Building JSONL for 400 rows
[PROGRESS] 10/400 requests written
[PROGRESS] 20/400 requests written
[PROGRESS] 30/400 requests written
[PROGRESS] 40/400 requests written
[PROGRESS] 50/400 requests written
[PROGRESS] 60/400 requests written
[PROGRESS] 70/400 requests written
[PROGRESS] 80/400 requests written
[PROGRESS] 90/400 requests written
[PROGRESS] 100/400 requests written
[PROGRESS] 110/400 requests written
[PROGRESS] 120/400 requests written
[PROGRESS] 130/400 requests written
[PROGRESS] 140/400 requests written
[PROGRESS] 150/400 requests written
[PROGRESS] 160/400 requests written
[PROGRESS] 170/400 requests written
[PROGRESS] 180/400 requests written
[PROGRESS] 190/400 requests written
[PROGRESS] 200/400 requests written
[PROGRESS] 210/400 requests written
[PROGRESS] 220/400 requests written
[PROGRESS] 230/400 requests written
[PROGRESS] 240/400 requests written
[PROGRESS] 250/400 requests written
[PROGRESS] 260/400 requests written
[PROGRESS] 270/400 requests written
[

getting final dataset for code file context

In [14]:
import json
import pandas as pd


# ============================================================
# Configuration
# ============================================================

RAW_BATCH_FILE = "infile_context_relevance_batch_results.jsonl"
INPUT_DATA_FILE = "../../data/df_n1.csv"
OUTPUT_DATA_FILE = "infile_context_relevance_results_n2_1.csv"


# ============================================================
# Parse model output
# ============================================================

def parse_context_output(text):

    if text is None:
        return ""

    return str(text).strip()


# ============================================================
# Extract model output from one Batch API result
# ============================================================

def extract_batch_output(batch_item):

    try:
        return (
            batch_item["response"]
            ["body"]
            ["choices"][0]
            ["message"]
            ["content"]
        )

    except (KeyError, IndexError, TypeError):
        return None


# ============================================================
# Process raw batch results
# ============================================================

def process_raw_batch_results(df, raw_batch_file):

    print("[START] Reading raw batch results...")

    # --------------------------------------------------------
    # Read JSONL
    # --------------------------------------------------------

    batch_results = []

    with open(
        raw_batch_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:

                batch_results.append(
                    json.loads(line)
                )

            except json.JSONDecodeError as e:

                print(
                    f"[WARN] Invalid JSON on line "
                    f"{line_number}: {e}"
                )

    print(
        f"[INFO] Loaded {len(batch_results)} "
        f"batch results"
    )

    print(
        f"[INFO] Original dataframe has "
        f"{len(df)} rows"
    )


    # --------------------------------------------------------
    # Match results using custom_id
    # --------------------------------------------------------

    results_by_custom_id = {
        str(item.get("custom_id")): item
        for item in batch_results
        if item.get("custom_id") is not None
    }


    # --------------------------------------------------------
    # Process every dataframe row
    # --------------------------------------------------------

    preds = []

    for index, row in df.iterrows():

        print(
            f"[ROW {index + 1}/{len(df)}] Processing"
        )

        # ----------------------------------------------------
        # Same skip condition as original API pipeline
        # ----------------------------------------------------

        if (
            pd.isna(row["hunk"])
            or pd.isna(row["oldf"])
        ):

            preds.append(
                {
                    "relevant_context": ""
                }
            )

            continue


        # ----------------------------------------------------
        # Find corresponding batch result
        # ----------------------------------------------------

        custom_id = f"row_{index}"

        batch_item = results_by_custom_id.get(
            custom_id
        )


        # ----------------------------------------------------
        # Fallback to positional matching
        # ----------------------------------------------------

        if (
            batch_item is None
            and index < len(batch_results)
        ):

            batch_item = batch_results[index]


        # ----------------------------------------------------
        # Missing result
        # ----------------------------------------------------

        if batch_item is None:

            print(
                f"[WARN] No batch result found "
                f"for row {index}"
            )

            preds.append(
                {
                    "relevant_context": ""
                }
            )

            continue


        # ----------------------------------------------------
        # Extract raw model output
        # ----------------------------------------------------

        raw_text = extract_batch_output(
            batch_item
        )


        # ----------------------------------------------------
        # Same parsing as original API code
        # ----------------------------------------------------

        context = parse_context_output(
            raw_text
        )


        preds.append(
            {
                "relevant_context": context
            }
        )


    # --------------------------------------------------------
    # Build final dataframe
    # --------------------------------------------------------

    print("\n[DONE] Building dataframe")

    pred_df = pd.DataFrame(
        preds
    )

    df = df.reset_index(
        drop=True
    )

    result_df = pd.concat(
        [
            df,
            pred_df
        ],
        axis=1
    )


    print(
        f"[DONE] Final dataframe shape: "
        f"{result_df.shape}"
    )

    return result_df


# ============================================================
# Run
# ============================================================

df = pd.read_csv(
    INPUT_DATA_FILE
)

result_df = process_raw_batch_results(
    df,
    RAW_BATCH_FILE
)


# ============================================================
# Save
# ============================================================

result_df.to_csv(
    OUTPUT_DATA_FILE,
    index=False
)

print(
    f"[SAVED] {OUTPUT_DATA_FILE}"
)

[START] Reading raw batch results...
[INFO] Loaded 400 batch results
[INFO] Original dataframe has 400 rows
[ROW 1/400] Processing
[ROW 2/400] Processing
[ROW 3/400] Processing
[ROW 4/400] Processing
[ROW 5/400] Processing
[ROW 6/400] Processing
[ROW 7/400] Processing
[ROW 8/400] Processing
[ROW 9/400] Processing
[ROW 10/400] Processing
[ROW 11/400] Processing
[ROW 12/400] Processing
[ROW 13/400] Processing
[ROW 14/400] Processing
[ROW 15/400] Processing
[ROW 16/400] Processing
[ROW 17/400] Processing
[ROW 18/400] Processing
[ROW 19/400] Processing
[ROW 20/400] Processing
[ROW 21/400] Processing
[ROW 22/400] Processing
[ROW 23/400] Processing
[ROW 24/400] Processing
[ROW 25/400] Processing
[ROW 26/400] Processing
[ROW 27/400] Processing
[ROW 28/400] Processing
[ROW 29/400] Processing
[ROW 30/400] Processing
[ROW 31/400] Processing
[ROW 32/400] Processing
[ROW 33/400] Processing
[ROW 34/400] Processing
[ROW 35/400] Processing
[ROW 36/400] Processing
[ROW 37/400] Processing
[ROW 38/400] 

In [15]:

df_relevant_context = result_df[
    ["patch_id", "relevant_context"]
]


In [16]:
df_relevant_context.head(20)

,patch_id,relevant_context
0,P000000,char *cmd;\n\tif(worker_command != NULL){\n\t\...
1,P000001,type Params struct {\n\tProjectID string\n\tRe...
2,P000002,"import { Component, enqueueRender } from '../c..."
3,P000003,"package libkbfs\n\nimport (\n\t""time""\n\n\t""gi..."
4,P000004,func (ai *AttendedInstaller) initializeUI() (e...
5,P000005,typedef struct {\n RpmostreedTransaction pare...
6,P000006,type diffRowItr struct {\n\tad *diff.Asyn...
7,P000007,import org.apache.calcite.rel.type.RelDataType...
8,P000008,using System;\nusing Iesi.Collections.Generic;...
9,P000009,"<ol class=""breadcrumb"">\n <li><%= link_to(das..."


In [17]:
df_relevant_context.to_csv('../../data/df_n2_1.csv',index=False)